# Validación Externa: Kermany → RSNA
### Entrena en Kermany (pediátrico, China) · Evalúa en RSNA (adulto, EE.UU.)

Responde a la crítica de severidad 10/10: *"no se demuestra que el método generalice a otros hospitales, escáneres o poblaciones"*.

**Diseño:**
1. Entrena el ensamblado (top-3) **únicamente** con Kermany.
2. Evalúa el mismo modelo, sin re-entrenar ni ajustar, sobre RSNA.
3. El modelo **nunca ve** una imagen de RSNA durante el entrenamiento.

**Diferencias entre dominios (por eso es una prueba dura):**
| | Kermany (entrenamiento) | RSNA (validación externa) |
|---|---|---|
| Población | Pediátrica (1–5 años) | Adulta |
| Origen | Guangzhou, China | NIH / EE.UU. |
| Formato | JPEG | DICOM (se convierte) |
| Etiqueta positiva | Neumonía diagnosticada | Opacidad pulmonar |

**⚠️ ANTES DE EJECUTAR:** debes aceptar las reglas de la competencia RSNA en Kaggle:
https://www.kaggle.com/c/rsna-pneumonia-detection-challenge/rules → botón "I Understand and Accept"

Sin ese paso, la descarga fallará con error 403.

**Expectativa honesta:** se espera una caída de exactitud (probablemente a 60–85%). Eso es normal, esperado, y ES el hallazgo a reportar.

In [ ]:
!pip install -q kaggle timm scikit-learn pydicom
import torch
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available())

In [ ]:
from google.colab import files
import os
print('Sube tu kaggle.json:')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
for fn in uploaded.keys(): os.rename(fn, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('OK')

## 1. Dataset de ENTRENAMIENTO: Kermany (interno)

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/data --unzip

import shutil, random
from pathlib import Path
random.seed(42)
SRC=Path('/content/data/chest_xray')
ALL={'NORMAL':[],'PNEUMONIA':[]}
for split in ['train','test','val']:
    sd=SRC/split
    if not sd.exists(): continue
    for img in (sd/'NORMAL').glob('*.jpeg'): ALL['NORMAL'].append(img)
    for img in (sd/'PNEUMONIA').glob('*.jpeg'): ALL['PNEUMONIA'].append(img)

DEST=Path('/content/kermany')
if DEST.exists(): shutil.rmtree(DEST)
for cls,imgs in ALL.items():
    imgs=imgs.copy(); random.shuffle(imgs)
    n=len(imgs); ntr=int(n*0.70); nv=int(n*0.15)
    for sn,si in {'train':imgs[:ntr],'val':imgs[ntr:ntr+nv],'test':imgs[ntr+nv:]}.items():
        od=DEST/sn/cls; od.mkdir(parents=True,exist_ok=True)
        for img in si: shutil.copy(img,od/img.name)
print('Kermany (interno) listo:')
for sn in ['train','val','test']:
    print(f'  {sn}:', {c: len(list((DEST/sn/c).glob("*"))) for c in ALL})

## 2. Dataset EXTERNO: RSNA

Descarga los DICOM y las etiquetas. Filtra a dos clases limpias:
- **Normal** (clase `Normal` en detailed_class_info)
- **Neumonía** (clase `Lung Opacity`)

Se **descarta** la clase ambigua `No Lung Opacity / Not Normal` (pacientes con otras patologías), porque el modelo nunca fue entrenado para reconocerla y su inclusión haría el resultado no interpretable.

In [ ]:
# Requiere haber aceptado las reglas en: kaggle.com/c/rsna-pneumonia-detection-challenge/rules
!kaggle competitions download -c rsna-pneumonia-detection-challenge -p /content/rsna_raw
!cd /content/rsna_raw && unzip -q -o rsna-pneumonia-detection-challenge.zip -d /content/rsna_raw
!ls /content/rsna_raw | head

In [ ]:
import pandas as pd
info = pd.read_csv('/content/rsna_raw/stage_2_detailed_class_info.csv')
print('Distribución de clases en RSNA:')
print(info['class'].value_counts())
print()
# Nos quedamos SOLO con las dos clases interpretables
info_u = info.drop_duplicates(subset='patientId')
normal_ids = info_u[info_u['class']=='Normal']['patientId'].tolist()
pneu_ids   = info_u[info_u['class']=='Lung Opacity']['patientId'].tolist()
print(f'Normal: {len(normal_ids)} | Lung Opacity (neumonía): {len(pneu_ids)}')
print('Descartada la clase ambigua "No Lung Opacity / Not Normal"')

In [ ]:
# Convertir DICOM -> PNG. Muestreamos un subconjunto balanceado para que sea manejable.
import pydicom, numpy as np
from PIL import Image
from pathlib import Path
random.seed(42)

N_PER_CLASS = 1000  # subconjunto balanceado (1000 normal + 1000 neumonía = 2000 imágenes)
sel_normal = random.sample(normal_ids, min(N_PER_CLASS, len(normal_ids)))
sel_pneu   = random.sample(pneu_ids,   min(N_PER_CLASS, len(pneu_ids)))

RSNA_DIR = Path('/content/rsna_external')
if RSNA_DIR.exists(): shutil.rmtree(RSNA_DIR)
(RSNA_DIR/'NORMAL').mkdir(parents=True, exist_ok=True)
(RSNA_DIR/'PNEUMONIA').mkdir(parents=True, exist_ok=True)

DICOM_DIR = Path('/content/rsna_raw/stage_2_train_images')

def dicom_to_png(pid, out_dir):
    try:
        ds = pydicom.dcmread(str(DICOM_DIR/f'{pid}.dcm'))
        arr = ds.pixel_array.astype(float)
        arr = (arr - arr.min())/(arr.max()-arr.min()+1e-8)*255.0
        img = Image.fromarray(arr.astype(np.uint8)).convert('RGB')
        img.save(out_dir/f'{pid}.png')
        return True
    except Exception as e:
        return False

ok_n = sum(dicom_to_png(p, RSNA_DIR/'NORMAL') for p in sel_normal)
ok_p = sum(dicom_to_png(p, RSNA_DIR/'PNEUMONIA') for p in sel_pneu)
print(f'Convertidas -> NORMAL: {ok_n} | PNEUMONIA: {ok_p}')
print(f'Total conjunto externo RSNA: {ok_n+ok_p} imágenes')

## 3. Entrenar en Kermany (el modelo NUNCA ve RSNA)

In [ ]:
import timm, time, json
import torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42)
NUM_CLASSES=2; BATCH=32; EPOCHS=10; LR=1e-4; PATIENCE=4
SIZES={'DenseNet121':224,'InceptionV3':299,'VGG16':224}  # top-3 del experimento principal

def loaders_kermany(img_size):
    train_tf=transforms.Compose([transforms.Resize((img_size,img_size)),transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(10),transforms.ColorJitter(brightness=0.15,contrast=0.15),
        transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    eval_tf=transforms.Compose([transforms.Resize((img_size,img_size)),transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    tr=datasets.ImageFolder('/content/kermany/train',transform=train_tf)
    va=datasets.ImageFolder('/content/kermany/val',transform=eval_tf)
    te=datasets.ImageFolder('/content/kermany/test',transform=eval_tf)
    return (DataLoader(tr,batch_size=BATCH,shuffle=True,num_workers=2),
            DataLoader(va,batch_size=BATCH,shuffle=False,num_workers=2),
            DataLoader(te,batch_size=BATCH,shuffle=False,num_workers=2), tr.classes)

def loader_rsna(img_size):
    eval_tf=transforms.Compose([transforms.Resize((img_size,img_size)),transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    ds=datasets.ImageFolder('/content/rsna_external',transform=eval_tf)
    return DataLoader(ds,batch_size=BATCH,shuffle=False,num_workers=2), ds.classes

def train_kermany(name, timm_name, img_size):
    tr,va,te,classes=loaders_kermany(img_size)
    if timm_name=='inception_v3':
        m=timm.create_model(timm_name,pretrained=True,num_classes=NUM_CLASSES,aux_logits=False).to(device)
    else:
        m=timm.create_model(timm_name,pretrained=True,num_classes=NUM_CLASSES).to(device)
    crit=nn.CrossEntropyLoss(); opt=optim.AdamW(m.parameters(),lr=LR,weight_decay=0.01)
    sch=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
    best,bstate,noimp=0.0,None,0
    print(f'\n--- Entrenando {name} en KERMANY ---')
    for ep in range(EPOCHS):
        t0=time.time(); m.train()
        for x,y in tr:
            x,y=x.to(device),y.to(device)
            opt.zero_grad(); crit(m(x),y).backward(); opt.step()
        sch.step(); m.eval(); vp,vt=[],[]
        with torch.no_grad():
            for x,y in va:
                vp.extend(m(x.to(device)).argmax(1).cpu().numpy()); vt.extend(y.numpy())
        vacc=accuracy_score(vt,vp)
        print(f'  Ep {ep+1}/{EPOCHS} val_acc={vacc:.4f} ({time.time()-t0:.0f}s)')
        if vacc>best: best=vacc; bstate={k:v.cpu().clone() for k,v in m.state_dict().items()}; noimp=0
        else:
            noimp+=1
            if noimp>=PATIENCE: print(f'  Early stop ep {ep+1}'); break
    if bstate: m.load_state_dict(bstate)
    return m, te, classes

def get_probs(model, loader):
    model.eval(); probs,labels=[],[]
    with torch.no_grad():
        for x,y in loader:
            p=torch.softmax(model(x.to(device)),1).cpu().numpy()
            probs.extend(p); labels.extend(y.numpy())
    return np.array(probs), np.array(labels)

def metrics(labels, preds, tag):
    acc=accuracy_score(labels,preds); prec=precision_score(labels,preds,average='macro',zero_division=0)
    rec=recall_score(labels,preds,average='macro',zero_division=0); f1=f1_score(labels,preds,average='macro',zero_division=0)
    cm=confusion_matrix(labels,preds); tn,fp=cm[0,0],cm[0,1]
    spec=tn/(tn+fp) if (tn+fp)>0 else 0
    print(f'\n[{tag}] acc={acc*100:.2f}% prec={prec*100:.2f}% sens={rec*100:.2f}% spec={spec*100:.2f}% f1={f1*100:.2f}%')
    return {'accuracy':acc,'precision':prec,'recall_sensitivity':rec,'f1':f1,
            'specificity':float(spec),'confusion_matrix':cm.tolist()}

In [ ]:
# Entrena los 3 mejores y guarda probabilidades en AMBOS conjuntos
MODELS={'DenseNet121':'densenet121','InceptionV3':'inception_v3','VGG16':'vgg16'}
probs_int, probs_ext = {}, {}
labels_int = labels_ext = None
resultados = {}

for name,tname in MODELS.items():
    m, te_loader, classes = train_kermany(name, tname, SIZES[name])
    # Interno (Kermany test)
    pi, li = get_probs(m, te_loader)
    probs_int[name]=pi
    if labels_int is None: labels_int=li
    resultados[f'{name} (interno)'] = metrics(li, pi.argmax(1), f'{name} INTERNO (Kermany)')
    # Externo (RSNA) - el modelo NUNCA vio estas imagenes
    rsna_loader, rsna_classes = loader_rsna(SIZES[name])
    pe, le = get_probs(m, rsna_loader)
    probs_ext[name]=pe
    if labels_ext is None: labels_ext=le
    resultados[f'{name} (externo)'] = metrics(le, pe.argmax(1), f'{name} EXTERNO (RSNA)')
    del m; torch.cuda.empty_cache()

print('\nOrden de clases Kermany:', classes)
print('Orden de clases RSNA:', rsna_classes)
print('(Deben coincidir: NORMAL=0, PNEUMONIA=1)')

## 4. Ensamblado: interno vs. externo

In [ ]:
# Ensamblado por promedio simple de los 3 (sin optimizar pesos en el externo, para no sesgar)
ens_int = np.mean([probs_int[n] for n in MODELS], axis=0)
ens_ext = np.mean([probs_ext[n] for n in MODELS], axis=0)

resultados['Ensamblado (interno)'] = metrics(labels_int, ens_int.argmax(1), 'ENSAMBLADO INTERNO (Kermany)')
resultados['Ensamblado (externo)'] = metrics(labels_ext, ens_ext.argmax(1), 'ENSAMBLADO EXTERNO (RSNA)')

print('\n' + '='*60)
print('REPORTE DETALLADO - ENSAMBLADO EN RSNA (EXTERNO)')
print('='*60)
print(classification_report(labels_ext, ens_ext.argmax(1), target_names=['NORMAL','PNEUMONIA'], zero_division=0))

In [ ]:
# Tabla comparativa interno vs externo + brecha de generalización
rows=[]
for name in list(MODELS)+['Ensamblado']:
    ai=resultados[f'{name} (interno)']['accuracy']*100
    ae=resultados[f'{name} (externo)']['accuracy']*100
    rows.append({'Modelo':name,
                 'Interno Kermany (%)':round(ai,2),
                 'Externo RSNA (%)':round(ae,2),
                 'Brecha (pts)':round(ae-ai,2)})
df=pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv('/content/validacion_externa.csv',index=False)
with open('/content/validacion_externa.json','w') as f: json.dump(resultados,f,indent=2)

print('\n=== INTERPRETACIÓN ===')
gap = rows[-1]['Brecha (pts)']
print(f'Brecha de generalización del ensamblado: {gap:+.2f} puntos porcentuales.')
print('Una caída es ESPERADA: RSNA es población adulta, otro país, otro etiquetado.')
print('Este número cuantifica honestamente los límites de generalización del modelo.')

files.download('/content/validacion_externa.csv')
files.download('/content/validacion_externa.json')
print('\nPasame los 2 archivos.')